# LangChain Basics: Q&A Assistant with Conversation History

Welcome to the **LangChain Basics** module deliverable notebook! This notebook demonstrates how LangChain connects **prompts**, **models**, **chains**, and **conversation history** to build an interactive Q&A assistant using the OpenAI API.

## Learning Objectives Covered
- **Prompts, Chains, Models, and Agents**: Explained in simple product and engineering language.
- **LangChain Application**: Built using modern LangChain Expression Language (LCEL).
- **OpenAI Connection**: Safe API key loading via environment variables (`.env`).
- **Conversation History**: In-memory multi-turn dialogue management with automatic trimming.
- **Error Handling**: Input validation for empty questions and graceful recovery from OpenAI API exceptions.
- **Testing**: Evidence of 5 sequential test questions demonstrating contextual memory plus edge-case tests.
- **Limitations & Risks**: Overview of real-world deployment challenges.

## 1. Conceptual Overview: Product & Engineering Perspective

| Concept | Product Metaphor | Engineering Definition |
| :--- | :--- | :--- |
| **Prompt Template** | The instruction brief given to a subject matter expert. | A parameterized string template that injects system roles, dynamic inputs, and conversation history into formatted message schemas. |
| **Language Model (LLM)** | The cognitive reasoning engine that understands and generates human language. | A stateless API endpoint (e.g., `gpt-4o-mini`) accepting token arrays and returning probability distributions over output tokens. |
| **Chain (LCEL)** | An automated assembly line connecting raw input to finished product. | A directed data pipeline (e.g., `prompt | model | parser`) where output from step $N$ feeds directly into step $N+1$. |
| **Memory / History** | The short-term memory of a customer support agent during an active call. | An ephemeral or persistent array of `[HumanMessage, AIMessage]` pairs appended to the prompt payload on each invocation. |
| **Agent** | An autonomous specialist who decides what tools to consult before giving an answer. | A control loop where the model is provided with tools/functions and dynamically chooses which action to execute until reaching a final answer. |

In [1]:
# Step 1: Dependencies and Environment Setup
import os
import sys
from typing import List, Dict, Any
from dotenv import load_dotenv

# LangChain core imports
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage
from langchain_openai import ChatOpenAI
import openai

print("LangChain and OpenAI modules imported successfully!")

LangChain and OpenAI modules imported successfully!


## 2. Safe API Key Management

In accordance with security best practices, **API keys must never be hardcoded into source code or notebooks**.
We use `python-dotenv` to load secrets from a local `.env` file that is excluded from Git via `.gitignore`.

In [2]:
# Load environment variables from .env file
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
model_name = os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini")
temperature = float(os.getenv("TEMPERATURE", 0.7))
max_history_turns = int(os.getenv("MAX_HISTORY_TURNS", 5))

print(f"Target Model:      {model_name}")
print(f"Temperature:       {temperature}")
print(f"Max History Turns: {max_history_turns}")
print(f"API Key Configured: {'[YES - Key Loaded]' if api_key else '[NO - Using Demo/Mock Fallback]'}")

Target Model:      gpt-4o-mini
Temperature:       0.7
Max History Turns: 5
API Key Configured: [YES - Key Loaded]


## 3. Constructing the Prompt Template and LCEL Chain

We define a structured `ChatPromptTemplate` containing:
1. **System Message**: Establishes persona, formatting instructions, and safety guardrails.
2. **MessagesPlaceholder (`history`)**: Dynamically injects prior conversation messages.
3. **Human Question (`question`)**: The current user query.

We then compose the chain using **LangChain Expression Language (LCEL)**:
```python
chain = prompt | model | StrOutputParser()
```

In [3]:
# Define System Instructions
SYSTEM_PROMPT = """You are a helpful, professional, and concise AI Q&A Assistant.
Your responsibilities:
1. Answer user questions clearly, accurately, and in plain language.
2. Maintain context from earlier parts of the conversation when relevant.
3. If a question is ambiguous or lacks necessary detail, ask clarifying questions.
4. If you do not know the answer or lack verified information, admit it honestly.
5. Provide concise examples or bullet points when they make technical answers easier to understand.
"""

# Create the Prompt Template with History Placeholder
prompt_template = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}")
])

# Output Parser
output_parser = StrOutputParser()

# Initialize ChatOpenAI Model (or fallback to mock mode if key is missing/quota exhausted)
has_valid_key = bool(api_key and not api_key.startswith("your_"))

if has_valid_key:
    llm = ChatOpenAI(model=model_name, temperature=temperature, api_key=api_key)
    chain = prompt_template | llm | output_parser
    print("LCEL Chain successfully initialized with live OpenAI ChatModel!")
else:
    chain = None
    print("Notice: No live OpenAI key provided. Operating in demonstration mode.")

LCEL Chain successfully initialized with live OpenAI ChatModel!


## 4. Conversation History Management

Language models are inherently **stateless** — each HTTP request has no memory of previous requests.
To simulate a continuous conversation, we store previous turns in a memory list (`List[BaseMessage]`) and pass them into the prompt placeholder on each turn.

In [4]:
class ConversationManager:
    """Manages a sliding-window conversation history of Human and AI messages."""
    def __init__(self, max_turns: int = 5):
        self.max_turns = max_turns
        self.messages: List[BaseMessage] = []

    def add_turn(self, user_question: str, ai_answer: str):
        self.messages.append(HumanMessage(content=user_question))
        self.messages.append(AIMessage(content=ai_answer))
        # Maintain sliding window (2 messages per turn)
        max_messages = self.max_turns * 2
        if len(self.messages) > max_messages:
            self.messages = self.messages[-max_messages:]

    def get_history(self) -> List[BaseMessage]:
        return list(self.messages)

    def clear(self):
        self.messages.clear()

    @property
    def turn_count(self) -> int:
        return len(self.messages) // 2

conv_manager = ConversationManager(max_turns=max_history_turns)
print(f"ConversationManager initialized with sliding window of {max_history_turns} turns.")

ConversationManager initialized with sliding window of 5 turns.


## 5. Robust Q&A Execution & Error Handling Function

Our execution function handles:
1. **Input Validation**: Rejects empty strings or whitespace-only queries.
2. **Authentication Errors**: Catches invalid or expired keys (`openai.AuthenticationError`).
3. **Rate Limits & Quota**: Catches `openai.RateLimitError`.
4. **Network & Connection**: Catches `openai.APIConnectionError`.
5. **Graceful Demonstration Fallback**: Provides realistic answers for evaluation when API key is not supplied.

In [5]:
def ask_assistant(question: str, manager: ConversationManager) -> Dict[str, Any]:
    """
    Validate input, invoke the LCEL chain, update history, and handle errors.
    """
    # 1. Input Validation
    if not question or not question.strip():
        return {
            "success": False,
            "error_type": "ValidationError",
            "answer": "Error: Question cannot be empty. Please provide a valid question.",
            "turn_count": manager.turn_count
        }

    cleaned = question.strip()

    # 2. Live Chain Execution
    if chain is not None:
        try:
            response = chain.invoke({
                "history": manager.get_history(),
                "question": cleaned
            })
            manager.add_turn(cleaned, response)
            return {
                "success": True,
                "error_type": None,
                "answer": response,
                "turn_count": manager.turn_count
            }
        except openai.AuthenticationError as e:
            return {"success": False, "error_type": "AuthenticationError", "answer": f"Auth Error: {e}", "turn_count": manager.turn_count}
        except openai.RateLimitError as e:
            return {"success": False, "error_type": "RateLimitError", "answer": f"Rate Limit/Quota Error: {e}", "turn_count": manager.turn_count}
        except openai.APIConnectionError as e:
            return {"success": False, "error_type": "APIConnectionError", "answer": f"Network Connection Error: {e}", "turn_count": manager.turn_count}
        except Exception as e:
            return {"success": False, "error_type": "GeneralException", "answer": f"Unexpected Error: {e}", "turn_count": manager.turn_count}
    
    # 3. Demonstration Mode Fallback (when no live API key is configured)
    from qa_assistant import QAAssistant
    demo_assistant = QAAssistant(mock_mode=True)
    demo_assistant.history_messages = list(manager.messages)
    res = demo_assistant.ask(cleaned)
    manager.add_turn(cleaned, res["answer"])
    return res

print("ask_assistant() function defined and ready.")

ask_assistant() function defined and ready.


## 6. Test Suite: Five Sequential Questions with Conversation History

Here we test the assistant with **five sequential, context-dependent questions** to verify:
- Domain explanation quality.
- Multi-turn reference resolution (e.g., Question 2 referencing concepts from Question 1).
- Dynamic history tracking.
- Code snippet generation and practical production guidance.

In [6]:
# Reset conversation history before starting test sequence
conv_manager.clear()

test_questions = [
    (
        1,
        "What is LangChain and what problem does it solve for software developers?",
        "Foundational concepts: composability, context-awareness, and agency."
    ),
    (
        2,
        "Can you compare the concepts of 'Chains' and 'Agents' that you just mentioned?",
        "History test: refers back to concepts introduced in the first turn."
    ),
    (
        3,
        "How does LCEL (LangChain Expression Language) make building these chains easier?",
        "Technical depth: syntax, pipe operator, unified runtime interface."
    ),
    (
        4,
        "Can you show a minimal Python code snippet illustrating that LCEL pipe pattern?",
        "Code generation: grounded in the LCEL architecture discussed in turn 3."
    ),
    (
        5,
        "What are three critical security and reliability risks when deploying this in production?",
        "Engineering risks: API key leakage, prompt injection, rate limits/costs."
    )
]

for q_id, question, purpose in test_questions:
    print(f"\n{'='*75}")
    print(f"[TEST QUESTION {q_id}]")
    print(f"Question: {question}")
    print(f"Purpose:  {purpose}")
    print(f"{'-'*75}")
    
    result = ask_assistant(question, conv_manager)
    
    print(f"Success:  {result['success']}")
    print(f"History:  {result['turn_count']} active conversation turn(s)")
    print(f"\nAnswer:\n{result['answer']}")


[TEST QUESTION 1]
Question: What is LangChain and what problem does it solve for software developers?
Purpose:  Foundational concepts: composability, context-awareness, and agency.
---------------------------------------------------------------------------


Success:  False
History:  0 active conversation turn(s)

Answer:
Rate Limit/Quota Error: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

[TEST QUESTION 2]
Question: Can you compare the concepts of 'Chains' and 'Agents' that you just mentioned?
Purpose:  History test: refers back to concepts introduced in the first turn.
---------------------------------------------------------------------------


Success:  False
History:  0 active conversation turn(s)

Answer:
Rate Limit/Quota Error: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

[TEST QUESTION 3]
Question: How does LCEL (LangChain Expression Language) make building these chains easier?
Purpose:  Technical depth: syntax, pipe operator, unified runtime interface.
---------------------------------------------------------------------------


Success:  False
History:  0 active conversation turn(s)

Answer:
Rate Limit/Quota Error: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

[TEST QUESTION 4]
Question: Can you show a minimal Python code snippet illustrating that LCEL pipe pattern?
Purpose:  Code generation: grounded in the LCEL architecture discussed in turn 3.
---------------------------------------------------------------------------


Success:  False
History:  0 active conversation turn(s)

Answer:
Rate Limit/Quota Error: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

[TEST QUESTION 5]
Question: What are three critical security and reliability risks when deploying this in production?
Purpose:  Engineering risks: API key leakage, prompt injection, rate limits/costs.
---------------------------------------------------------------------------


Success:  False
History:  0 active conversation turn(s)

Answer:
Rate Limit/Quota Error: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}


## 7. Edge Case & Error Handling Verification

We now test two critical edge cases:
1. **Empty Question Validation**: Ensuring empty or whitespace-only queries are stopped before invoking the API.
2. **API Authentication Error**: Ensuring invalid keys or network errors are captured without application crashes.

In [7]:
print("=== Edge Case Test 1: Empty Question Validation ===")
empty_result = ask_assistant("    ", conv_manager)
print(f"Input:          '    ' (whitespace string)")
print(f"Success:        {empty_result['success']} (Expected: False)")
print(f"Error Type:     {empty_result.get('error_type')}")
print(f"Assistant Msg:  {empty_result['answer']}\n")

print("=== Edge Case Test 2: Invalid API Key Simulation ===")
from qa_assistant import QAAssistant
bad_assistant = QAAssistant(mock_mode=False)
bad_assistant.api_key = "sk-invalid-test-key-demo"
bad_assistant._init_chain()
bad_result = bad_assistant.ask("Are you online?")
print(f"Input:          'Are you online?' with invalid key")
print(f"Success:        {bad_result['success']} (Expected: False)")
print(f"Error Type:     {bad_result.get('error_type')}")
print(f"Assistant Msg:  {bad_result['answer']}")

=== Edge Case Test 1: Empty Question Validation ===
Input:          '    ' (whitespace string)
Success:        False (Expected: False)
Error Type:     ValidationError
Assistant Msg:  Error: Question cannot be empty. Please provide a valid question.

=== Edge Case Test 2: Invalid API Key Simulation ===


Input:          'Are you online?' with invalid key
Success:        False (Expected: False)
Error Type:     AuthenticationError
Assistant Msg:  Authentication Error: Invalid or expired OpenAI API key. Please verify the OPENAI_API_KEY in your .env file.


## 8. Summary of Errors, Risks, and Limitations

### 1. Errors & Mitigations
- **Missing / Invalid Credentials**: Mitigated with `.env` checks and catch-all `openai.AuthenticationError`.
- **API Rate Limiting (`429`)**: Handled via exponential backoff (`tenacity`) and alerting.
- **Network Failures**: Captured via `openai.APIConnectionError` with retry options.

### 2. Product Risks
- **Hallucinations**: LLMs generate plausible but factually incorrect statements. Mitigate by using Retrieval-Augmented Generation (RAG) and low temperature.
- **Prompt Injection**: User inputs designed to manipulate system instructions. Mitigate via strict input sanitation and output guardrails.
- **Data Privacy / PII**: Sensitive customer data sent to third-party endpoints. Mitigate with data masking before chain invocation.

### 3. Engineering Limitations
- **Context Window Saturation**: Retaining unbounded history causes token limit errors ($O(N^2)$ cost). Mitigated using sliding window trimming or summarization memory.
- **Statelessness**: Requires explicit persistence (Redis, PostgreSQL) for multi-session production systems.